# 1kg_eur — 0.1 % MAF GRM (Batch)

Genomic relatedness matrix over **all variants with MAF > 0.1 %** — no upper bound.
This is a *superset* of the 1 % panel (`04_grm_panel_qc.ipynb`) and the 5 % panel
(`grm_5pct_shards.ipynb`), so it is built from the raw unified panel PGEN rather than
from either of those BEDs.

Together the three form a nested MAF ladder: 0.1 % ⊃ 1 % ⊃ 5 %.

QC is stricter than the 1 % panel (tighter call rate and HWE), since lower-frequency
variants are more genotyping-error prone. **Note:** if the ladder is meant to isolate the
effect of the MAF threshold alone, set `GENO_MAX`/`HWE_P` below to match nb04
(`0.05` / `1e-6`) so QC is held constant across rungs.

**Step 1** — one Batch task: download unified panel PGEN, plink2 QC + MAF filter, write BED + freq.

**Step 2** — sharded Batch tasks compute the GRM in parallel.

The two stay separate jobs on purpose: the filter needs a cheap high-disk machine, the shards
need a very large-memory one, and the shard machine size cannot be known until step 1 reports
the output BED size.

## Config

In [ ]:
import math, subprocess

PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS  = f"{WS_GS}/1kg_eur"

# ── input: raw unified panel (NOT nb04 output — this panel is a superset) ──────
PANEL_GS = f"{WS_GS}/01_ancestry_filtering/unified_panel/unified_panel_v9"
KEEP_GS  = f"{R_GS}/01_ancestry/round2/1kg_eur_keep_ids.txt"   # pre-rename filename

# ── plink binaries staged in GCS ──────────────────────────────────────────────
PLINK2_BIN_GS = f"{R_GS}/03_grm/bin/plink2"
PLINK_BIN_GS  = f"{R_GS}/03_grm/bin/plink"    # plink 1.9, staged by 06_grm_shards

# ── QC thresholds ─────────────────────────────────────────────────────────────
MAF_MIN  = 0.001     # 0.1 %, no upper bound
GENO_MAX = 0.01      # stricter than nb04 (0.05)
HWE_P    = "1e-10"   # stricter than nb04 (1e-6)

# ── output ────────────────────────────────────────────────────────────────────
BED_NAME      = "1kg_CEUGBR_GRM_0p1pct"
FILTER_OUT_GS = f"{R_GS}/03_grm/grm_input_0p1pct"
SHARD_OUT_GS  = f"{R_GS}/03_grm/shards_0p1pct"
LOG_GS        = f"{R_GS}/03_grm/logs_0p1pct"

# ── step 1 sizing (matches the validated nb02 shape for this same panel) ──────
FILTER_MACHINE = "n1-highmem-16"
FILTER_DISK_GB = 750     # 116 GB pgen in + BED out (possibly 200 GB+) + working

# ── step 2 sizing ─────────────────────────────────────────────────────────────
# Do NOT hand-edit BED_SIZE_GB — the autosize cell below sets it from the real
# file in GCS once step 1 has finished.
BED_SIZE_GB   = None
SHARD_DISK_GB = 400
N_SHARDS      = 20
N_TASKS       = 5

print(f"MAF > {MAF_MIN} (no upper bound), geno < {GENO_MAX}, HWE p > {HWE_P}")
print(f"output: {FILTER_OUT_GS}/{BED_NAME}.bed")

## Stage plink binaries

Copies the locally installed plink2 / plink 1.9 to GCS so Batch workers can use them.
Safe to re-run: skips anything already staged.

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
stage () {{
  local LOCAL_BIN="$1" GS_PATH="$2"
  if gcloud storage ls "$GS_PATH" >/dev/null 2>&1; then
    echo "already staged: $GS_PATH"
  elif [ -x "$LOCAL_BIN" ]; then
    gcloud storage cp "$LOCAL_BIN" "$GS_PATH" && echo "staged: $GS_PATH"
  else
    echo "MISSING $LOCAL_BIN — run 04_grm_panel_qc (plink2) / 06_grm_shards (plink) first"
    exit 1
  fi
}}
stage "$HOME/bin/plink2" "{PLINK2_BIN_GS}"
stage "$HOME/bin/plink"  "{PLINK_BIN_GS}"
"""], check=True)

## Install dsub

In [ ]:
subprocess.run(["bash", "-c", f"""
pip install --quiet --upgrade 'dsub>=0.5.3'
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"
echo "dsub: $(dsub --version)"
"""], check=True)

## Step 1: QC and filter to MAF > 0.1 %

Downloads the unified panel inside the job (116 GB) rather than via dsub localization,
so a missing input fails fast and visibly in the job log.

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "{PROJECT_ID}" --regions "{REGION}" \
  --logging "{LOG_GS}" \
  --service-account "{SERVICE_ACCOUNT}" \
  --network "{NETWORK}" --subnetwork "{SUBNETWORK}" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "grm-0p1pct-filter" \
  --machine-type "{FILTER_MACHINE}" --disk-size "{FILTER_DISK_GB}" \
  --input  PLINK2="{PLINK2_BIN_GS}" \
  --input  PLINK="{PLINK_BIN_GS}" \
  --input  KEEP="{KEEP_GS}" \
  --output-recursive OUT="{FILTER_OUT_GS}" \
  --command '
    set -eo pipefail
    chmod +x "$PLINK2" "$PLINK"
    mkdir -p /mnt/data/panel

    echo "=== downloading unified panel ==="
    for EXT in pgen pvar psam; do
      gcloud storage cp "{PANEL_GS}.$EXT" "/mnt/data/panel/panel.$EXT"
    done
    ls -lh /mnt/data/panel

    echo "=== plink2 QC: MAF > {MAF_MIN}, geno < {GENO_MAX}, HWE > {HWE_P} ==="
    "$PLINK2" \
      --pfile /mnt/data/panel/panel \
      --keep "$KEEP" --nonfounders \
      --maf {MAF_MIN} \
      --geno {GENO_MAX} \
      --hwe {HWE_P} keep-fewhet \
      --max-alleles 2 --rm-dup exclude-all \
      --threads $(nproc) \
      --make-bed --out "$OUT/{BED_NAME}"

    echo "=== allele frequencies (plink 1.9 .frq for --read-freq) ==="
    "$PLINK" --bfile "$OUT/{BED_NAME}" --freq --out "$OUT/{BED_NAME}_freq"

    echo "=== summary ==="
    echo "variants: $(wc -l < "$OUT/{BED_NAME}.bim")"
    echo "samples:  $(wc -l < "$OUT/{BED_NAME}.fam")"
    du -h "$OUT/{BED_NAME}.bed"
  ' 2>&1 | tee /tmp/filter_0p1.log
tail -3 /tmp/filter_0p1.log
"""], check=True)

## Monitor step 1

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "{PROJECT_ID}" --location "{REGION}" \
  --jobs "grm-0p1pct-filter*" --status '*' --full 2>&1 | tail -40
"""], check=False)

## Autosize the shard machine

Run after step 1 succeeds. Reads the real BED size from GCS and derives the shard
machine type — no hand-editing. Re-run if you change `N_SHARDS`.

In [ ]:
N1_MAX_MEM_MB = 624 * 1024   # largest n1 memory footprint available
N1_MAX_VCPUS  = 96

_du = subprocess.run(
    ["gcloud", "storage", "du", "-s", f"{FILTER_OUT_GS}/{BED_NAME}.bed"],
    capture_output=True, text=True)
if _du.returncode != 0 or not _du.stdout.split():
    raise SystemExit(f"BED not found — has step 1 finished?\n{_du.stderr.strip()}")

BED_SIZE_GB = int(_du.stdout.split()[0]) / 1024**3
MEMORY_MB   = math.ceil((BED_SIZE_GB * 2 + 4) * 1024 / 256) * 256

SHARD_VCPUS = 16
while MEMORY_MB / SHARD_VCPUS > 8192 and SHARD_VCPUS < N1_MAX_VCPUS:
    SHARD_VCPUS += 16

SHARD_MACHINE = (f"n1-custom-{SHARD_VCPUS}-{MEMORY_MB}-ext"
                 if MEMORY_MB / SHARD_VCPUS > 6656
                 else f"n1-custom-{SHARD_VCPUS}-{MEMORY_MB}")
PLINK_MEM_MB = MEMORY_MB - 8192

print(f"BED size:      {BED_SIZE_GB:.1f} GB")
print(f"memory:        {MEMORY_MB} MB ({MEMORY_MB/1024:.0f} GB)")
print(f"machine:       {SHARD_MACHINE}")
print(f"plink --memory {PLINK_MEM_MB}")

if MEMORY_MB > N1_MAX_MEM_MB:
    print(f"\n*** {MEMORY_MB/1024:.0f} GB exceeds the ~624 GB n1 ceiling. Options:")
    print("      - raise N_SHARDS (smaller GRM slice per task) and re-run this cell")
    print("      - switch to a memory-optimized family (m1-ultramem-*)")
    print("      - drop to the 1 % panel for this rung of the ladder")

## Step 2: Build shard task list

In [ ]:
shards_per_task = N_SHARDS // N_TASKS
tasks_tsv = "/tmp/grm_0p1pct_tasks.tsv"
with open(tasks_tsv, "w") as fh:
    fh.write("--env TASK_SHARDS\n")
    for t in range(N_TASKS):
        fh.write(",".join(str(s) for s in
                 range(t * shards_per_task + 1, (t + 1) * shards_per_task + 1)) + "\n")
print(open(tasks_tsv).read())

## Submit GRM shard jobs

Requires the autosize cell above to have run.

In [ ]:
assert BED_SIZE_GB is not None, "run the autosize cell first"

subprocess.run(["bash", "-c", f"""
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "{PROJECT_ID}" --regions "{REGION}" \
  --logging "{LOG_GS}" \
  --service-account "{SERVICE_ACCOUNT}" \
  --network "{NETWORK}" --subnetwork "{SUBNETWORK}" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "grm-0p1pct-shards" \
  --machine-type "{SHARD_MACHINE}" --disk-size "{SHARD_DISK_GB}" \
  --input            PLINK_BIN="{PLINK_BIN_GS}" \
  --input-recursive  BED_DIR="{FILTER_OUT_GS}" \
  --output-recursive SHARD_DIR="{SHARD_OUT_GS}" \
  --tasks /tmp/grm_0p1pct_tasks.tsv \
  --command '
    set -eo pipefail
    chmod +x "$PLINK_BIN"
    IFS="," read -ra SHARDS <<< "$TASK_SHARDS"
    for k in "${{SHARDS[@]}}"; do
      echo "--- shard $k / {N_SHARDS} ---"
      "$PLINK_BIN" \
        --bfile "$BED_DIR/{BED_NAME}" \
        --read-freq "$BED_DIR/{BED_NAME}_freq.frq" \
        --make-grm-bin --parallel "$k" {N_SHARDS} \
        --memory {PLINK_MEM_MB} \
        --out "$SHARD_DIR/grm.shard$k"
      rm -f "$SHARD_DIR/grm.shard$k.grm.N.bin"
    done
  ' 2>&1 | tee /tmp/grm_0p1pct_jobs.log
"""], check=True)

## Monitor shard jobs

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "{PROJECT_ID}" --location "{REGION}" \
  --jobs "grm-0p1pct-shards*" --status '*' --full 2>&1 | tail -40
"""], check=False)

## Copy notebook to bucket

In [ ]:
import os
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/grm_0p1pct_shards.ipynb')
subprocess.run(['gcloud', 'storage', 'cp', _nb,
                f'{SHARD_OUT_GS}/notebooks/grm_0p1pct_shards.ipynb'], check=True)
print('copied')